In [2]:
from enola.enola import Enola
import qiskit # for generic circuits with dependency. Convert the gates in circuits into hardware-supported gate set
from animation import CodeGen

In [3]:
import os

# Specify the directory
directory = 'benchmarks/'

# Get the list of files (excluding directories)
file_paths = sorted([os.path.join(directory, f) for f in os.listdir(directory) if os.path.isfile(os.path.join(directory, f))])

# Print or return the list of file names
print(file_paths)

['benchmarks/4gt13_92.qasm', 'benchmarks/4mod5-v1_22.qasm', 'benchmarks/adder.qasm', 'benchmarks/barenco_tof_4.qasm', 'benchmarks/barenco_tof_5.qasm', 'benchmarks/mod5mils_65.qasm', 'benchmarks/mod_mult_55.qasm', 'benchmarks/or.qasm', 'benchmarks/qaoa5.qasm', 'benchmarks/queko_05_0.qasm', 'benchmarks/queko_10_3.qasm', 'benchmarks/queko_15_1.qasm', 'benchmarks/rc_adder_6.qasm', 'benchmarks/tof_4.qasm', 'benchmarks/tof_5.qasm', 'benchmarks/toffoli.qasm', 'benchmarks/vbe_adder_3.qasm']


In [4]:
# generic_circuit_name = "benchmarks/4gt13_92.qasm"
# generic circuits with dependencies
# extract two-qubit gates from the circuit

def generate_list_for_circuit(generic_circuit_name):
    list_gate_two_qubit = []
    with open(generic_circuit_name, 'r') as f:
        qasm_str = f.read()
        circuit = qiskit.QuantumCircuit.from_qasm_str(qasm_str)
        cz_circuit = qiskit.transpile(circuit, basis_gates=["cz", "id", "u2", "u1", "u3"])
        instruction = cz_circuit.data
        for ins in instruction:
            if ins.operation.num_qubits == 2:
                list_gate_two_qubit.append((ins.qubits[0]._index, ins.qubits[1]._index))
    return list_gate_two_qubit

gates_for_each_circuit = []
for file in file_paths:
    gates_for_each_circuit.append(generate_list_for_circuit(file))

In [8]:
commutable_circuit_name = 'my_custom_circuit'
default_compiler = Enola(
        name=commutable_circuit_name,
        dir='./results/',
        reverse_to_initial=True,
        full_code=True)

commutable_circuit_name = 'my_custom_circuit_25_qubits'
default_compiler_25 = Enola(
        name=commutable_circuit_name,
        dir='./results/',
        reverse_to_initial=True,
        full_code=True
)

commutable_circuit_name = '90_qubits_circuit'
default_compiler_90 = Enola(
        name=commutable_circuit_name,
        dir='./results/',
        reverse_to_initial=True,
        full_code=True
)


In [141]:
circuits = []
for file_path in file_paths:
    file_name = os.path.basename(file_path)
    new_file_name = file_name.replace('.qasm', '')
    circuit = Enola(
        name=new_file_name,
        dir='./results/',
        trivial_layout=True,
        dependency=True,
        routing_strategy="maximalis",
        reverse_to_initial=True,
        use_window=True,
        full_code=True
    )
    circuits.append(circuit)

In [49]:
# an example of setting an architecture with an SLM of 10 columns and rows, and an AOD with 10 columns and rows
default_compiler.setArchitecture([10, 10, 10, 10])

default_compiler_25.setArchitecture([20, 20, 20, 20])

default_compiler_90.setArchitecture([90, 90, 90, 90])

In [142]:
for circuit in circuits:
    circuit.setArchitecture([20, 20, 20, 20])

In [50]:
# Initialize the graph edge list with stages
graph_edge_list = [
    # Stage 1
    [0, 2], [1, 3],
    # Stage 2
    [2, 4], [3, 5],
    # Stage 3
    [4, 6], [5, 7]
]

graph_edge_list_25 = [
   # Stage 1
    [0, 1], [2, 3], [4, 5],
    # Stage 2
    [1, 6], [3, 7], [5, 8],
    # Stage 3
    [6, 9], [7, 10], [8, 11],
    # Stage 4
    [9, 12], [10, 13], [11, 14],
    # Stage 5
    [12, 15], [13, 16], [14, 17]
]

graph_edge_list_90 = [
    # Stage 1
    [0, 1], [2, 3], [4, 5], [6, 7], [8, 9], [10, 11], [12, 13], [14, 15], [16, 17], [18, 19], [20, 21],
    # Stage 2
    [1, 22], [3, 23], [5, 24], [7, 25], [9, 26], [11, 27], [13, 28], [15, 29], [17, 30], [19, 31], [21, 32],
    # Stage 3
    [22, 33], [23, 34], [24, 35], [25, 36], [26, 37], [27, 38], [28, 39], [29, 40], [30, 41], [31, 42], [32, 43],
    # Stage 4
    [33, 44], [34, 45], [35, 46], [36, 47], [37, 48], [38, 49], [39, 50], [40, 51], [41, 52], [42, 53], [43, 54],
    # Stage 5
    [44, 55], [45, 56], [46, 57], [47, 58], [48, 59], [49, 60], [50, 61], [51, 62], [52, 63], [53, 64], [54, 65],
    # Stage 6
    [55, 66], [56, 67], [57, 68], [58, 69], [59, 70], [60, 71], [61, 72], [62, 73], [63, 74], [64, 75], [65, 76]
]


default_compiler.setProgram(graph_edge_list)
default_compiler_25.setProgram(graph_edge_list_25)
default_compiler_90.setProgram(graph_edge_list_90)

In [143]:
for i, circuit in enumerate(circuits):
    circuit.setProgram(gates_for_each_circuit[i])
    print(circuit.g_q)

[(2, 3), (0, 4), (1, 4), (0, 4), (0, 1), (1, 4), (0, 4), (0, 1), (1, 4), (2, 4), (3, 4), (2, 3), (2, 4), (3, 4), (2, 3), (2, 3), (1, 4), (0, 4), (0, 1), (1, 4), (0, 4), (0, 1), (1, 4), (2, 4), (3, 4), (2, 3), (2, 4), (3, 4), (2, 3), (0, 4)]
[(0, 2), (1, 3), (2, 3), (3, 4), (2, 4), (2, 3), (3, 4), (2, 4), (2, 3), (2, 3), (3, 4)]
[(0, 1), (2, 3), (2, 3), (1, 2), (0, 3), (0, 1), (0, 1), (2, 3), (2, 3), (0, 3)]
[(5, 6), (3, 6), (5, 6), (3, 5), (3, 5), (4, 5), (2, 5), (4, 5), (2, 4), (2, 4), (1, 4), (0, 4), (1, 4), (0, 4), (4, 5), (2, 5), (4, 5), (5, 6), (3, 6), (5, 6), (3, 5), (3, 5), (4, 5), (2, 5), (4, 5), (1, 4), (0, 4), (1, 4), (0, 4), (4, 5), (2, 5), (4, 5), (2, 4), (2, 4)]
[(7, 8), (4, 8), (7, 8), (4, 7), (4, 7), (6, 7), (3, 7), (6, 7), (3, 6), (3, 6), (5, 6), (2, 6), (5, 6), (2, 5), (2, 5), (1, 5), (0, 5), (1, 5), (0, 5), (5, 6), (2, 6), (5, 6), (6, 7), (3, 7), (6, 7), (7, 8), (4, 8), (7, 8), (4, 7), (4, 7), (6, 7), (3, 7), (6, 7), (5, 6), (2, 6), (5, 6), (1, 5), (0, 5), (1, 5), (0,

In [51]:
# program_list_qaoa = default_compiler.solve(save_file=True)

# print(program_list_qaoa)
# print(default_compiler_25.solve(save_file=True))

var_string = default_compiler_90.solve(save_file=True)
print(var_string)

[INFO] Enola: Start Solving
[INFO] Enola: Run scheduling
[INFO] Enola: Time for scheduling: 0.001058816909790039s
[INFO] Enola: Start SA-based placement
[INFO] Enola: SA-Based Placer: Iter 0, cost: 63.803125
[INFO] Enola: Time for placement: 34.16747999191284s
[INFO] Enola: Solve for Rydberg stage 2/3.
[INFO] Enola: Solve for Rydberg stage 3/3.
[INFO] Enola: Solve for Rydberg stage 4/3.
[INFO] Enola: Time for routing: 0.006776094436645508s
[INFO] Enola: Toal Time: 34.83480882644653s
[{'type': 'Init', 'name': 'Init', 'n_q': 77, 'x_high': 90, 'y_high': 90, 'c_high': 90, 'r_high': 90, 'duration': 24, 'slm_qubit_idx': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76], 'slm_qubit_xys': [(114, 165), (95, 165), (0, 120), (0, 105), (0, 0), (0, 15), 

In [144]:
print(circuits[0].g_q)
circuits[0].solve(save_file=True)

[(2, 3), (0, 4), (1, 4), (0, 4), (0, 1), (1, 4), (0, 4), (0, 1), (1, 4), (2, 4), (3, 4), (2, 3), (2, 4), (3, 4), (2, 3), (2, 3), (1, 4), (0, 4), (0, 1), (1, 4), (0, 4), (0, 1), (1, 4), (2, 4), (3, 4), (2, 3), (2, 4), (3, 4), (2, 3), (0, 4)]
[INFO] Enola: Start Solving
[INFO] Enola: Run scheduling
[INFO] Enola: Time for scheduling: 0.00011301040649414062s
20
[INFO] Enola: Time for placement: 1.2636184692382812e-05s
[INFO] Enola: Solve for Rydberg stage 2/26.
[INFO] Enola: Solve for Rydberg stage 3/26.
[INFO] Enola: Solve for Rydberg stage 4/26.
[INFO] Enola: Solve for Rydberg stage 5/26.
[INFO] Enola: Solve for Rydberg stage 6/26.
[INFO] Enola: Solve for Rydberg stage 7/26.
[INFO] Enola: Solve for Rydberg stage 8/26.
[INFO] Enola: Solve for Rydberg stage 9/26.
[INFO] Enola: Solve for Rydberg stage 10/26.
[INFO] Enola: Solve for Rydberg stage 11/26.
[INFO] Enola: Solve for Rydberg stage 12/26.
[INFO] Enola: Solve for Rydberg stage 13/26.
[INFO] Enola: Solve for Rydberg stage 14/26.
[INFO

[{'type': 'Init',
  'name': 'Init',
  'n_q': 5,
  'x_high': 20,
  'y_high': 20,
  'c_high': 20,
  'r_high': 20,
  'duration': 24,
  'slm_qubit_idx': [0, 1, 2, 3, 4],
  'slm_qubit_xys': [(0, 0), (19, 0), (38, 0), (57, 0), (76, 0)],
  'aod_qubit_idx': [],
  'aod_qubit_crs': [],
  'aod_col_act_idx': [],
  'aod_col_xs': [],
  'aod_row_act_idx': [],
  'aod_row_ys': [],
  'state': {},
  'all_slms': [(0, 0), (19, 0), (38, 0), (57, 0), (76, 0), (42, 0), (4, 0)]},
 {'type': 'Rydberg',
  'name': 'Rydberg_0:Rydberg',
  'gates': [],
  'duration': 0.36,
  'state': {}},
 {'type': 'Activate',
  'name': 'Reload_1:ReloadRow_0:Activate',
  'col_idx': [4],
  'col_xs': [76],
  'row_idx': [0],
  'row_ys': [0],
  'pickup_qs': [4],
  'duration': 15,
  'state': {}},
 {'type': 'Move',
  'name': 'Reload_1:ReloadRow_0:Parking:Move',
  'cols': [{'id': 4, 'shift': -2, 'begin': 76, 'end': 74}],
  'rows': [{'id': 0, 'shift': -2, 'begin': 0, 'end': -2}],
  'duration': 26.967994498529684,
  'state': {}},
 {'type': 'Mo

In [145]:
programs = []
for i in range(0, len(circuits)):
    program = circuits[i].solve(save_file=True)
    programs.append(program)


# for circuit in circuits:
#     circuit.solve(save_file=True)
# circuit = circuits[1]
# print(circuit.g_q)
# print(circuit.n_q)
# circuit.solve(save_file=True)

[INFO] Enola: Start Solving
[INFO] Enola: Run scheduling
[INFO] Enola: Time for scheduling: 7.700920104980469e-05s
20
[INFO] Enola: Time for placement: 6.9141387939453125e-06s
[INFO] Enola: Solve for Rydberg stage 2/26.
[INFO] Enola: Solve for Rydberg stage 3/26.
[INFO] Enola: Solve for Rydberg stage 4/26.
[INFO] Enola: Solve for Rydberg stage 5/26.
[INFO] Enola: Solve for Rydberg stage 6/26.
[INFO] Enola: Solve for Rydberg stage 7/26.
[INFO] Enola: Solve for Rydberg stage 8/26.
[INFO] Enola: Solve for Rydberg stage 9/26.
[INFO] Enola: Solve for Rydberg stage 10/26.
[INFO] Enola: Solve for Rydberg stage 11/26.
[INFO] Enola: Solve for Rydberg stage 12/26.
[INFO] Enola: Solve for Rydberg stage 13/26.
[INFO] Enola: Solve for Rydberg stage 14/26.
[INFO] Enola: Solve for Rydberg stage 15/26.
[INFO] Enola: Solve for Rydberg stage 16/26.
[INFO] Enola: Solve for Rydberg stage 17/26.
[INFO] Enola: Solve for Rydberg stage 18/26.
[INFO] Enola: Solve for Rydberg stage 19/26.
[INFO] Enola: Solve fo

In [146]:
# example command
import json
!python3 simulator.py results/code/my_custom_circuit_code_full.json

!python3 simulator.py results/code/my_custom_circuit_25_qubits_code_full.json

!python3 simulator.py results/code/90_qubits_circuit_code_full.json

string_to_add = "_code_full"

for file_path in file_paths:
    file_name = os.path.basename(file_path)
    new_file_name = file_name.replace('.qasm', f'{string_to_add}.json')
    !python3 simulator.py results/code/{new_file_name}

In [102]:
# import animator
from animation import Animator

# generate animation for qaoa circuit
# animator_1 = Animator(
#         "results/code/my_custom_circuit_code_full.json",
#         show_graph=True,
#         edges=graph_edge_list,
#         dir='./results/animations/'
#     )

# # generate animation for qaoa circuit
# animator_2 = Animator(
#         "results/code/my_custom_circuit_25_qubits_code_full.json",
#         show_graph=True,
#         edges=graph_edge_list_25,
#         dir='./results/animations/'
#     )


In [115]:
print(circuits[1].result_json)

{'name': '4mod5-v1_22', 'layers': []}


In [154]:


def calculate_total_duration(code):
    """
    Calculate the total execution time of the quantum circuit.

    Args:
        code (list): A list of instruction dictionaries, each containing a 'duration' key.
    
    Returns:
        float: Total execution time in seconds.
    """
    total_duration = 0  # Initialize total time in microseconds

    # Loop through the list of instructions
    for inst in code:
        total_duration += inst.get('duration', 0)  # Safely sum up durations, defaulting to 0 if 'duration' key is missing

    # Convert total duration from microseconds to seconds
    total_duration_seconds = total_duration / 1_000_000  # Convert µs to seconds

    return total_duration_seconds



def parse_execution_stages(data):
    """
    Parses the provided quantum execution data to extract which gates are executed 
    at each 'Rydberg' stage.

    Args:
    - data: A list of dictionaries where each dictionary represents a stage of quantum operations.

    Returns:
    - stages_with_gates: A dictionary where the keys are 'Rydberg' stage names and the values are lists of gates executed at that stage.
    """
    stages_with_gates = {}
    
    for stage in data:
        # Only process 'Rydberg' stages
        if stage.get('type') == 'Rydberg':
            stage_name = stage.get('name', 'Unnamed Stage')
            # Only process stages that contain gates
            if 'gates' in stage and stage['gates']:
                gates = stage['gates']
                # print(gates['id'])
                # print(gates)
                new_gates = []
                for gate in gates:
                    # print(gate['id'])
                    # print(gate['q0'], gate['q1'])
                    new_gates.append((gate['id'], (gate['q0'], gate['q1'])))
                    # new_gates[gate['id']] = [gate['q0'], gate['q1']]
                stages_with_gates[stage_name] = new_gates
    
    return stages_with_gates




   

def extract_rydberg_stages(program_list):
    rydberg_stages = [entry for entry in program_list if entry.get('type') == 'Rydberg']
    return rydberg_stages

for i in range(0, len(programs)):
    rydberg_stages = extract_rydberg_stages(programs[i])
    file_path = file_paths[i]
    print(file_path)
    print(len(rydberg_stages))




# print("total duration for circuit 1: ", duration_1)
# print("total duration for circuit 2: ", duration_2)




benchmarks/4gt13_92.qasm
27
benchmarks/4mod5-v1_22.qasm
11
benchmarks/adder.qasm
7
benchmarks/barenco_tof_4.qasm
35
benchmarks/barenco_tof_5.qasm
49
benchmarks/mod5mils_65.qasm
17
benchmarks/mod_mult_55.qasm
26
benchmarks/or.qasm
7
benchmarks/qaoa5.qasm
9
benchmarks/queko_05_0.qasm
5
benchmarks/queko_10_3.qasm
6
benchmarks/queko_15_1.qasm
10
benchmarks/rc_adder_6.qasm
41
benchmarks/tof_4.qasm
22
benchmarks/tof_5.qasm
29
benchmarks/toffoli.qasm
7
benchmarks/vbe_adder_3.qasm
34
